<a href="https://colab.research.google.com/github/BF667/UVRC/blob/main/UVRC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UVRC — Ultimate Vocal Remover Colab

1. **Install** → Run Cell 1
2. **Separate** → Pick a model from the dropdown, upload audio, run Cell 2
3. **Listen** → Run Cell 3 to play results side-by-side

In [ ]:
#@title 1️⃣ Install UVRC
%cd /content
!pip install -q git+https://github.com/BF667/UVRC.git
import os
os.makedirs('ckpts', exist_ok=True)
os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)
print('Installation complete.')

In [ ]:
#@title 📤 Upload Audio File

from google.colab import files
import os, shutil

uploaded = files.upload()
for fname in uploaded:
    dest = os.path.join('/content/input', fname)
    shutil.move(fname, dest)
    print(f'Uploaded: {fname}')

# List available input files
input_files = [f for f in os.listdir('/content/input') if f.lower().endswith(('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma'))]
if input_files:
    print(f'\nFiles in input/: {input_files}')
else:
    print('No audio files found in input/')

In [ ]:
#@title 🔗 Mount Google Drive
mount_drive = True #@param {type:"boolean"}
drive_path = '/content/drive' #@param {type:"string"}

if mount_drive:
    from google.colab import drive
    drive.mount(drive_path, force_remount=True)
    print(f'Drive mounted at {drive_path}')
else:
    print('Drive mount skipped.')

In [ ]:
#@title 2️⃣ Separate Audio
model_preset = 'VOCALS-MelBand-Roformer (by KimberleyJSN)' #@param ['VOCALS-InstVocHQ', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'VOCALS-MelBand-Roformer (by Becruily)', 'VOCALS-MelBand-Roformer voc_Fv5 (by Gabox)', 'VOCALS-MelBand-Roformer voc_gabox2 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv4 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv6 experimental (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv7 beta 3 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv7 beta 2 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv7 beta (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv3 (by Gabox)', 'VOCALS-MelBand-Roformer Kim FT (by Unwa)', 'VOCALS-MelBand-Roformer Kim FT 2 (by Unwa)', 'VOCALS-MelBand-Roformer Kim FT 2 Bleedless (by Unwa)', 'VOCALS-Mel-Roformer FT 3 Preview (by unwa)', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'INST-Mel-Roformer v1 (by gabox)', 'INST-Mel-Roformer v1e (by gabox)', 'INST-Mel-Roformer v2 (by gabox)', 'DRUMS-MelBand-Roformer (by Gabox)', 'DRUMS-MelBand-Roformer v2 (by Gabox)', 'DRUMS-MelBand-Roformer v2e (by Gabox)', 'DRUMS-BS-Roformer (by viperx)', 'DEREVERB-MelBand-Roformer (by Gabox)', 'DEREVERB-MelBand-Roformer v2 (by Gabox)', 'DENOISE-MelBand-Roformer Denoise v1 (by gabox)', 'DENOISE-MelBand-Roformer Denoise v2 (by gabox)', 'DENOISE-MelBand-Roformer Denoise v3 (by gabox)', 'DENOISE-MelBand-Roformer Denoise v4 (by gabox)', 'KARAOKE-MelBand-Roformer (by Gabox)', 'CROWD-MelBand-Roformer Crowd (by Gabox)', 'DENOISE-DEBLEED-MelBand-Roformer (by Unwa)', 'DENOISE-DEBLEED-MelBand-Roformer v2 (by Unwa)', 'DENOISE-DEBLEED-MelBand-Roformer (by Gabox)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'PHANTOM-CENTER-HTDemucs (by wesleyr36)', 'GUITAR-MelBand-Roformer (by becruily)'] {allow-input: true}
input_file = '/content/input/audio.mp3' #@param {type:"string"}
output_folder = '/content/output' #@param {type:"string"}
extract_instrumental = True #@param {type:"boolean"}
export_format = 'flac PCM_16' #@param ['wav FLOAT', 'flac PCM_16', 'flac PCM_24']
use_tta = False #@param {type:"boolean"}
overlap = 4 #@param {type:"slider", min:2, max:40, step:1}
chunk_size = '485100' #@param ['352800', '485100'] {allow-input: true}

import os
from UVRC.multi import resolve_model

# Resolve format
flac_file = export_format.startswith('flac')
pcm_type = export_format.split(' ')[1] if flac_file else None

# Validate input
if not os.path.isfile(input_file):
    print(f'Input file not found: {input_file}') 
    raise SystemExit(1)

# Resolve model from registry
result = resolve_model(model_preset, int(chunk_size), overlap)
if result is None:
    print(f'Unknown model: {model_preset}')
    raise SystemExit(1)

model_type, config_path, start_check_point = result
print(f'Model: {model_preset}')
print(f'Type: {model_type}')

# Build and run CLI
cmd = f"uvr-cli --model_type {model_type} --config_path '{config_path}' --start_check_point '{start_check_point}' --input_file '{input_file}' --store_dir '{output_folder}'"
if extract_instrumental:
    cmd += ' --extract_instrumental'
if flac_file:
    cmd += ' --flac_file'
if use_tta:
    cmd += ' --use_tta'
if pcm_type:
    cmd += f' --pcm_type {pcm_type}'

print(f'\nRunning separation...')
get_ipython().system(cmd)
print(f'\nDone! Output: {output_folder}')

In [ ]:
#@title 🔄 Batch Process Folder
batch_input = '/content/input' #@param {type:"string"}
batch_output = '/content/output' #@param {type:"string"}
batch_model = 'VOCALS-MelBand-Roformer (by KimberleyJSN)' #@param ['VOCALS-InstVocHQ', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'VOCALS-MelBand-Roformer (by Becruily)', 'VOCALS-MelBand-Roformer voc_Fv5 (by Gabox)', 'VOCALS-MelBand-Roformer voc_gabox2 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv4 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv6 experimental (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv7 beta 3 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv7 beta 2 (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv7 beta (by Gabox)', 'VOCALS-MelBand-Roformer voc_Fv3 (by Gabox)', 'VOCALS-MelBand-Roformer Kim FT (by Unwa)', 'VOCALS-MelBand-Roformer Kim FT 2 (by Unwa)', 'VOCALS-MelBand-Roformer Kim FT 2 Bleedless (by Unwa)', 'VOCALS-Mel-Roformer FT 3 Preview (by unwa)', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'INST-Mel-Roformer v1 (by gabox)', 'INST-Mel-Roformer v1e (by gabox)', 'INST-Mel-Roformer v2 (by gabox)', 'DRUMS-MelBand-Roformer (by Gabox)', 'DRUMS-MelBand-Roformer v2 (by Gabox)', 'DRUMS-MelBand-Roformer v2e (by Gabox)', 'DRUMS-BS-Roformer (by viperx)', 'DEREVERB-MelBand-Roformer (by Gabox)', 'DEREVERB-MelBand-Roformer v2 (by Gabox)', 'DENOISE-MelBand-Roformer Denoise v1 (by gabox)', 'DENOISE-MelBand-Roformer Denoise v2 (by gabox)', 'DENOISE-MelBand-Roformer Denoise v3 (by gabox)', 'DENOISE-MelBand-Roformer Denoise v4 (by gabox)', 'KARAOKE-MelBand-Roformer (by Gabox)', 'CROWD-MelBand-Roformer Crowd (by Gabox)', 'DENOISE-DEBLEED-MelBand-Roformer (by Unwa)', 'DENOISE-DEBLEED-MelBand-Roformer v2 (by Unwa)', 'DENOISE-DEBLEED-MelBand-Roformer (by Gabox)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'PHANTOM-CENTER-HTDemucs (by wesleyr36)', 'GUITAR-MelBand-Roformer (by becruily)'] {allow-input: true}
batch_extract_instr = True #@param {type:"boolean"}
batch_format = 'flac PCM_16' #@param ['wav FLOAT', 'flac PCM_16', 'flac PCM_24']

import os, glob
from UVRC.multi import resolve_model

flac_file = batch_format.startswith('flac')
pcm_type = batch_format.split(' ')[1] if flac_file else None

audio_exts = ('.mp3', '.wav', '.flac', '.ogg', '.m4a', '.aac', '.wma')
files = sorted([f for f in os.listdir(batch_input) if f.lower().endswith(audio_exts)])

if not files:
    print(f'No audio files found in {batch_input}')
    raise SystemExit(1)

result = resolve_model(batch_model)
if result is None:
    print(f'Unknown model: {batch_model}')
    raise SystemExit(1)

model_type, config_path, start_check_point = result
print(f'Batch processing {len(files)} file(s) with {batch_model}\n')

for i, fname in enumerate(files, 1):
    fpath = os.path.join(batch_input, fname)
    print(f'[{i}/{len(files)}] {fname}')
    cmd = f"uvr-cli --model_type {model_type} --config_path '{config_path}' --start_check_point '{start_check_point}' --input_file '{fpath}' --store_dir '{batch_output}'"
    if batch_extract_instr:
        cmd += ' --extract_instrumental'
    if flac_file:
        cmd += ' --flac_file'
    if pcm_type:
        cmd += f' --pcm_type {pcm_type}'
    get_ipython().system(cmd)

print(f'\nBatch complete! {len(files)} file(s) processed.')

In [ ]:
#@title 🎛️ Ensemble Multiple Results
ensemble_files_input = '' #@param {type:"string"}
ensemble_algorithm = 'avg_wave' #@param ['avg_wave', 'median_wave', 'min_wave', 'max_wave', 'avg_fft', 'median_fft', 'min_fft', 'max_fft']
ensemble_weights = '' #@param {type:"string"}
ensemble_output = '/content/output/ensemble_result.wav' #@param {type:"string"}

import subprocess, sys

files = [f.strip() for f in ensemble_files_input.split(',') if f.strip()]
if len(files) < 2:
    print('Need at least 2 files separated by commas.')
    raise SystemExit(1)

cmd = [sys.executable, '-m', 'UVRC.ensemble',
       '--files'] + files + [
       '--type', ensemble_algorithm,
       '--output', ensemble_output]

if ensemble_weights:
    weights = [float(w.strip()) for w in ensemble_weights.split(',') if w.strip()]
    if len(weights) != len(files):
        print(f'Weight count ({len(weights)}) must match file count ({len(files)}).')
        raise SystemExit(1)
    cmd += ['--weights'] + [str(w) for w in weights]

print(f'Ensembling {len(files)} files with {ensemble_algorithm}...')
subprocess.run(cmd)
print(f'Output: {ensemble_output}')

In [ ]:
#@title 3️⃣ Play Results
import os, glob
from IPython.display import Audio, display, HTML

output_files = sorted(
    glob.glob('/content/output/*.wav') + glob.glob('/content/output/*.flac'),
    key=os.path.getmtime,
    reverse=True
)

if not output_files:
    print('No output files found.')
else:
    for f in output_files:
        name = os.path.basename(f)
        size_mb = os.path.getsize(f) / 1e6
        display(HTML(f'<b>{name}</b> ({size_mb:.1f} MB)'))
        display(Audio(f))

In [ ]:
#@title 💾 Save to Google Drive
save_source = '/content/output' #@param {type:"string"}
save_dest = '/content/drive/MyDrive/UVRC_Output' #@param {type:"string"}

import os, shutil

os.makedirs(save_dest, exist_ok=True)
for f in os.listdir(save_source):
    if f.lower().endswith(('.wav', '.flac', '.mp3')):
        src = os.path.join(save_source, f)
        dst = os.path.join(save_dest, f)
        shutil.copy2(src, dst)
        print(f'Copied: {f}')

print(f'\nSaved to {save_dest}')